# Fisheye Visual Odometry

Implement a visual odometry pipeline for fisheye drone footage that tracks features with Lucas–Kanade flow and recovers a relative camera trajectory.

This notebook only provides **data paths** and **trajectory evaluation**. The pipeline itself is up to you.

**Deliverables** (all in this notebook):
- Recovered relative trajectory, compared against GT with the evaluation cells below
- All code cells run and plots rendered
- Overlay video
- Summary markdown cell at the end: implementation choices, parameter tuning, observed trajectory quality

In [ ]:
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from pose_helpers import (
    TrajectoryResult,
    TrajectoryVideoWriter,
    analyze_trajectory,
    draw_tracks,
    plot_trajectory,
    write_video_with_trajectory,
)

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11

In [ ]:
# --- Data paths ---
PROJECT_ROOT = Path("./flight")
VIDEO_PATH = PROJECT_ROOT / "recording.mp4"
CALIBRATION_PATH = PROJECT_ROOT / "calibration-params.json"
GT_PATH = PROJECT_ROOT / "flight_trajectory_coordinates.csv"

# --- Outputs ---
OVERLAY_VIDEO_PATH = Path("tracks_overlay.mp4")              # written by your pipeline: frames with your tracked points
TRAJECTORY_VIDEO_PATH = Path("tracks_with_trajectory.mp4")   # written by the evaluation: your overlay + trajectory plot

for p in (VIDEO_PATH, CALIBRATION_PATH, GT_PATH):
    if not p.exists():
        raise FileNotFoundError(p)

print(f"Video       : {VIDEO_PATH}")
print(f"Calibration : {CALIBRATION_PATH}")
print(f"Ground truth: {GT_PATH}")

## Your pipeline

Add as many cells as you need. The evaluation below expects two things.

**1. A `traj_result: TrajectoryResult`** with:

| Field | Shape | Meaning |
| :-- | :-- | :-- |
| `positions` | `(N + 1, ≥2)` array | Row 0 is the starting pose; row `i + 1` is the pose at `frame_indices[i]`. Only X and Y are evaluated. |
| `frame_indices` | `N` ints | Absolute frame index in `recording.mp4` of each recovered pose (used to match GT rows). |
| `inliers` | `N` ints | Inlier count of the motion estimate for each recovered pose. |
| `timestamps` | `N` floats | Time of each recovered pose, in seconds. |

**2. An overlay video at `OVERLAY_VIDEO_PATH`** with one frame per frame of `recording.mp4`, in the same order (frame `k` of the overlay = frame `k` of the recording). `write_video_with_trajectory` seeks into it by `frame_indices`. `TrajectoryVideoWriter` and `draw_tracks` from `pose_helpers` can do this for you.

In [ ]:
# Your pipeline here. It must produce `traj_result` and write OVERLAY_VIDEO_PATH.
traj_result: TrajectoryResult = ...

## Evaluation

Do not modify. Checks the result format, plots the trajectory and inliers, renders the trajectory video, and compares against GT.

In [ ]:
assert isinstance(traj_result, TrajectoryResult), "traj_result must be a TrajectoryResult"
n = len(traj_result.frame_indices)
assert n > 0, "No poses recovered"
assert traj_result.positions.shape[0] == n + 1, "positions must have len(frame_indices) + 1 rows"
assert len(traj_result.inliers) == n and len(traj_result.timestamps) == n, "inliers/timestamps must match frame_indices"
assert OVERLAY_VIDEO_PATH.exists(), f"Overlay video not found: {OVERLAY_VIDEO_PATH}"
print(f"Recovered {n} poses")

plot_trajectory(traj_result)

In [ ]:
write_video_with_trajectory(
    OVERLAY_VIDEO_PATH,
    traj_result,
    output_path=TRAJECTORY_VIDEO_PATH,
    stride=5,
)

In [ ]:
analyze_trajectory(traj_result, GT_PATH)

## Summary

**Implementation choices:** _..._

**Parameter tuning:** _which parameters you tuned, what you tried, and the evidence from your run that justified the final values._

**Trajectory quality and overlay video:** _how close the result is to GT, where it drifts or fails, and why._